# 02 Global metacells

First metacell pass, over the whole dataset. Input is the merged object written
by `01_preprocess.py`; output is a cell object carrying metacell assignments and
gene masks, plus the collected metacell object that `03` annotates.

Order of operations matters here: per-cell QC metrics are computed **before** any
gene is dropped, because percent mito needs the MT genes still present and
`exclude_genes` removes them.

Cell filtering is the composite excluded-gene-fraction filter plus a UMI floor.
MAD and miQC are implemented below but commented out, because stacking three
filters makes the attrition table hard to read. Scrublet lives in `01` and is off
by default.

Background: docs/04_building_metacells.md (mechanics), docs/05_gene_lists.md
(excluded / lateral / noisy), docs/06_parameters.md (every tunable below).

In [ ]:
#import libraries
import scanpy as sc
import anndata as ad
import metacells as mc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sb
import os
from scipy.stats import median_abs_deviation


In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
# Every path and tunable comes from config/config.yaml, so this notebook holds
# no sample ids and no absolute paths.
import os, yaml

CFG_DIR = os.path.abspath(os.path.join("..", "config"))
CFG  = yaml.safe_load(open(os.path.join(CFG_DIR, "config.yaml")))
ROOT = os.path.abspath("..")
rp = lambda p: p if os.path.isabs(p) else os.path.join(ROOT, p)

PROJECT  = CFG["project"]["name"]
h5ad_dir = rp(os.path.join(CFG["paths"]["h5ad"], "global", "metacell"))
qc_dir   = rp(os.path.join(CFG["paths"]["csvs"], "global", "qc", "metacell"))
os.makedirs(h5ad_dir, exist_ok=True); os.makedirs(qc_dir, exist_ok=True)

M = CFG["metacells_global"]
# target cells per metacell. Sets the graining level gamma = cells / metacells.
TARGET_METACELL_SIZE     = M["target_metacell_size"]
# target UMIs per metacell. Whichever of the two binds first is the one that acts.
TARGET_METACELL_UMIS     = M["target_metacell_umis"]
# metacells per pile: how the divide-and-conquer step chunks the data
TARGET_METACELLS_IN_PILE = M["target_metacells_in_pile"]
MIN_PILE                 = M["min_pile"]
MAX_PILE                 = M["max_pile"]
# hard UMI floor, a safety net rather than the main QC filter
MIN_UMI_FLOOR            = M["min_umi_floor"]
# a cell whose excluded genes (mito, hemoglobin, MALAT1) exceed this is dropped
MAX_EXCL_FRAC            = M["max_excluded_gene_fraction"]
RANDOM_SEED              = M["random_seed"]

QC_XLSX_PATH = os.path.join(qc_dir, f"{PROJECT}_global_metacell_QC.xlsx")

# MC2 parallelises over piles; take the scheduler's allocation when there is one
n_processors = int(os.environ.get("LSB_DJOB_NUMPROC", os.cpu_count() or 4))
mc.ut.set_processors_count(n_processors)
print(f"Using {n_processors} processors")


def write_xlsx_sheets(path, sheets):
    """Create-or-append sheets to an .xlsx workbook. Idempotent: a sheet of the
    same name is replaced, so a rerun of one chunk does not duplicate tabs."""
    mode = "a" if os.path.exists(path) else "w"
    kw = {"if_sheet_exists": "replace"} if mode == "a" else {}
    with pd.ExcelWriter(path, engine="openpyxl", mode=mode, **kw) as xw:
        for name, df in sheets.items():
            df.to_excel(xw, sheet_name=name[:31], index=False)
    print(f"Wrote {list(sheets)} -> {path}")


### Helper functions: MAD outliers, miQC mixture model


In [ ]:

# def is_outlier(values, nmads=5, normal_scale=False):
#     """
#     Boolean MAD-based outlier mask (both tails).
    
#     Default = RAW MAD with nmads=5, i.e. literally 5 median-absolute-deviations from
#     the median. This matches the current single-cell reference convention
#     (Heumos et al. 2023, Nat Rev Genet 24:550, the sc-best-practices `is_outlier`,
#     which calls scipy.median_abs_deviation with its default scale=1.0 and nmads=5).
    
#     Set normal_scale=True to switch to SD-EQUIVALENT units (scales the MAD by ~1.4826
#     so it estimates the SD under normality); that is the scater/scuttle convention
#     (McCarthy et al. 2017, Bioinformatics 33:1179), whose default is nmads=3.
#     Do NOT combine normal_scale=True with nmads=5 unless 5 SD-equivalents is meant
#     (a much wider, ~7.4 raw-MAD band that no standard uses).
#     """
#     values = np.asarray(values, float)
#     med = np.median(values)
#     mad = median_abs_deviation(values, scale=("normal" if normal_scale else 1.0))
#     if mad == 0:
#         return np.zeros_like(values, dtype=bool)
#     return (values < med - nmads * mad) | (values > med + nmads * mad)


# def fit_miqc(mito_percent, n_genes, prob_threshold=0.75, n_init=8, max_iter=300, seed=0):
#     """
#     miQC-style QC (Hippen et al. 2021, PLoS Comput Biol 17:e1009290), reimplemented in
#     Python. Fits a 2-component mixture of LINEAR REGRESSIONS of % mito on the number of
#     detected genes, via EM (miQC uses R/flexmix for the same model). The component with the
#     HIGHER INTERCEPT is the 'compromised' distribution. A cell is removed if its posterior
#     probability of being compromised exceeds `prob_threshold`. Safeguard (miQC's
#     keep_all_below_boundary=TRUE): never remove a cell sitting below the healthy line.

#     Returns: prob_compromised (array), keep (bool array), params (list), comp_idx (int).
#     """
#     rng = np.random.default_rng(seed)
#     x = np.asarray(n_genes, float)
#     y = np.asarray(mito_percent, float)
#     n = len(y)
#     X = np.column_stack([np.ones(n), x])          # design matrix [1, n_genes]
#     best = None
#     for _ in range(n_init):
#         r = rng.random(n)
#         resp = np.column_stack([r, 1.0 - r])
#         ll_old = -np.inf
#         params = None
#         for _it in range(max_iter):
#             #M-step: weighted least squares per component - refits line based on weighted least sq. means and 
#             #cells currently belonging to that line count more.
#             params = []
#             for k in range(2):
#                 w = resp[:, k] + 1e-9
#                 sw = np.sqrt(w)
#                 #beta= [intercept, slope], so X @ beta = mito %
#                 beta, *_ = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)
#                 #residuals
#                 res = y - X @ beta
#                 var = max(np.sum(w * res ** 2) / np.sum(w), 1e-6)
#                 #pi= overall share of the line
#                 pi = max(np.mean(resp[:, k]), 1e-6)
#                 params.append((beta, var, pi))
#                #E-step
#             #log Gaussian density needed to measure how close the cell is to Line 0/1, weighted by share of each line (pi)
#             logdens = np.empty((n, 2))
#             for k, (beta, var, pi) in enumerate(params):
#                 res = y - X @ beta
#                 logdens[:, k] = np.log(pi) - 0.5 * np.log(2 * np.pi * var) - 0.5 * res ** 2 / var
#             m = logdens.max(axis=1, keepdims=True)
#             ll = float(np.sum(m.ravel() + np.log(np.exp(logdens - m).sum(axis=1))))
#             resp = np.exp(logdens - m)
#             resp /= resp.sum(axis=1, keepdims=True)
#             if ll - ll_old < 1e-6:
#                 break
#             ll_old = ll
#         if best is None or ll > best[0]:
#             best = (ll, params, resp.copy())
#     _, params, resp = best
#     intercepts = [p[0][0] for p in params]
#     comp = int(np.argmax(intercepts))             # compromised = higher intercept
#     healthy = 1 - comp
#     prob_comp = resp[:, comp]
#     keep = prob_comp < prob_threshold
#     pred_healthy = X @ params[healthy][0]
#     keep = keep | (y <= pred_healthy)              # keep_all_below_boundary means cells =< healthy cutoff are retained irrespective of which line it lands near
#     return prob_comp, keep, params, comp


# def write_xlsx_sheets(path, sheets):
#     """Create-or-append sheets to an .xlsx workbook (idempotent; replaces same-named tabs)."""
#     if os.path.exists(path):
#         with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
#             for name, df in sheets.items():
#                 df.to_excel(xw, sheet_name=name[:31], index=False)
#     else:
#         with pd.ExcelWriter(path, engine="openpyxl", mode="w") as xw:
#             for name, df in sheets.items():
#                 df.to_excel(xw, sheet_name=name[:31], index=False)
#     print(f"Wrote {list(sheets)} -> {path}")


## 1. Load the merged object


In [ ]:
# written by 01
merged_path = os.path.join(h5ad_dir, f"{PROJECT}.merged_samples.h5ad")
merged = ad.read_h5ad(merged_path)
mc.ut.set_name(merged, f"{PROJECT}.cells")
print("Shape:", merged.shape, "| cells:", merged.n_obs, "| genes:", merged.n_vars)
print(merged.obs["sample_origin"].value_counts())


## 2. Per-cell QC metrics 

`total_umis`, `n_genes`, and `percent.mt` are all needed downstream 


In [ ]:
total_umis = np.asarray(merged.X.sum(axis=1)).flatten()
n_genes    = np.asarray((merged.X > 0).sum(axis=1)).flatten()

## Using mito_mask  = merged.var_names.str.startswith("MT-") | merged.var_names.str.startswith("MTRNR"): 
##MTRNR clause adds only the nuclear-encoded MTRNR2L humanin-like pseudogenes, which are not mitochondrial. 
##Including them inflates percent_mito.

mito_mask  = merged.var_names.str.startswith("MT-") 
mito_umis  = np.asarray(merged[:, mito_mask].X.sum(axis=1)).flatten()
percent_mito = 100.0 * mito_umis / np.maximum(total_umis, 1)   # 0-100 scale

merged.obs["total_umis"]   = total_umis
merged.obs["n_genes"]      = n_genes
merged.obs["percent_mito"] = percent_mito

for label, v in [("total_umis", total_umis), ("n_genes", n_genes), ("percent_mito", percent_mito)]:
    qs = np.percentile(v, [1, 5, 25, 50, 75, 95, 99])
    print(label, "P1/5/25/50/75/95/99:", np.round(qs, 2))


## 3. MAD-based cell-QC (per sample)

flag cells that deviate by more than `MAD_NMADS` median-absolute-deviations from their own median on `log10(total_umis)` and `log10(n_genes)`


In [ ]:
# merged.obs["mad_outlier"] = False
# for s, idx in merged.obs.groupby("sample_origin", observed=True).groups.items():
#     sub = merged.obs.loc[idx]
#     out = (is_outlier(np.log10(sub["total_umis"] + 1), MAD_NMADS) |
#            is_outlier(np.log10(sub["n_genes"] + 1),   MAD_NMADS))
#     merged.obs.loc[idx, "mad_outlier"] = out

# #hard low floor for safety, 500
# merged.obs.loc[merged.obs["total_umis"] < MIN_UMI_FLOOR, "mad_outlier"] = True

# print("MAD outliers per sample:")
# print(merged.obs.groupby("sample_origin", observed=True)["mad_outlier"].mean().mul(100).round(2))
# print(f"\nTotal MAD-flagged: {int(merged.obs['mad_outlier'].sum())} "
#       f"({100*merged.obs['mad_outlier'].mean():.2f}%)")

# fig, ax = plt.subplots(1, 2, figsize=(11, 4))
# ax[0].hist(np.log10(merged.obs['total_umis'] + 1), bins=100); ax[0].set_xlabel("log10 total_umis")
# ax[1].hist(np.log10(merged.obs['n_genes'] + 1),   bins=100); ax[1].set_xlabel("log10 n_genes")
# plt.tight_layout(); plt.show()


## 4. miQC mitochondrial filtering

Fit the two-component miQC mixture model on (`percent.mt` ~ `n_genes`) and remove cells whose
posterior probability of being compromised exceeds `MIQC_PROB`. 

Inspect the plot: the red line is the fitted **compromised** distribution, the green line the **healthy** one; removed cells should
be the high-mito arm. (Fit here is global)


In [ ]:
# prob_comp, keep, params, comp = fit_miqc(
#     merged.obs["percent_mito"].values, merged.obs["n_genes"].values,
#     prob_threshold=MIQC_PROB, seed=RANDOM_SEED,
# )
# merged.obs["miqc_prob_compromised"] = prob_comp
# merged.obs["miqc_compromised"] = ~keep

# print(f"miQC flags {int((~keep).sum())} cells ({100*np.mean(~keep):.2f}%) as compromised")

# xx = np.linspace(merged.obs["n_genes"].min(), merged.obs["n_genes"].max(), 100)
# plt.figure(figsize=(6.5, 5))
# plt.scatter(merged.obs["n_genes"], merged.obs["percent_mito"], s=2,
#             c=np.where(keep, "#4E9E2F", "#8B1A1A"), alpha=0.3)
# for k, col, lab in [(comp, "red", "compromised"), (1 - comp, "green", "healthy")]:
#     b = params[k][0]; plt.plot(xx, b[0] + b[1] * xx, col, lw=2, label=lab)
# plt.xlabel("n_genes detected"); plt.ylabel("% mito"); plt.legend()
# plt.title("miQC model (green = keep arm, red = remove arm)"); plt.show()


## 5. Excluded / lateral / noisy gene lists

Three lists, three jobs:

- **excluded**: dropped from the object entirely (artefact and QC genes)
- **lateral**: still counted and still used for outlier detection, but not used
  to compute cell-cell similarity. Real biology that should not define identity.
- **noisy**: extra slack in outlier detection for bursty genes. Orthogonal to
  lateral.

Which gene belongs on which list, and why the TCR/BCR V/J segments matter more on
a subset than they do here: docs/05_gene_lists.md.

In [ ]:
Excluded_gnames = [
    "MALAT1", "NEAT1",                     
    "HBB", "HBA1", "HBA2", "HBD", "HBM",   
    "MT-ND1","MT-ND2","MT-CO1","MT-CO2","MT-ATP8","MT-ATP6","MT-CO3","MT-ND3","MT-ND4L",
    "MT-ND4","MT-ND5","MT-ND6","MT-CYB",
    "MTRNR2L11","MTRNR2L12","MTRNR2L13","MTRNR2L6","MTRNR2L10","MTRNR2L8","MTRNR2L7",
    "MTRNR2L5","MTRNR2L4","MTRNR2L1","MTRNR2L3",
]

#dropped the old 'MT1.*' pattern, it matched metallothioneins (MT1A/MT2A) - which are stress-response genes
Excluded_gpatterns = [
    "^MT-.*",     
    "^MTRNR.*",   
]

#Cell cycle (Tirosh 2016 Science 352:189 = Seurat cc.genes/regev_lab list)
CC_S = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL",
        "PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP",
        "CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN",
        "DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
CC_G2M = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B",
          "MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1",
          "KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C",
          "KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA",
          "PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
# Stress/immediate-early/dissociation (van den Brink 2017 Nat Methods 14:935; O'Flanagan 2019 Genome Biol 20:210)
STRESS = ["FOS","FOSB","JUN","JUNB","JUND","EGR1","IER2","ATF3","ZFP36","SOCS3","NR4A1","DUSP1",
          "PPP1R15A","HSPA1A","HSPA1B","HSPA6","HSPB1","HSP90AA1","HSP90AB1","DNAJB1","DNAJA1","BAG3","UBC"]
#Sex-linked
SEX = ["XIST","RPS4Y1","DDX3Y","UTY","KDM5D","EIF1AY","NLGN4Y","USP9Y","ZFY"]
#Platelet/megakaryocyte ambient
PLATELET = ["PPBP","PF4","GP9","GP1BA","ITGA2B","TUBB1","NRGN","CAVIN2","GNG11"]

Lateral_gnames = (CC_S + CC_G2M + STRESS + SEX + PLATELET +
    ["HLA-F","HLA-G","HLA-A","HLA-E","HLA-C","HLA-B","HLA-DRB5","HLA-DRB1","HLA-DQA1","HLA-DQB1",
     "HLA-DQB1-AS1","HLA-DQA2","HLA-DQB2","HLA-DOB","HLA-DMB","HLA-DMA","HLA-DOA","HLA-DPA1",
     "HLA-DPB1","JCHAIN"])
Lateral_gpatterns = [
    # (Brief Funct Genomics 2022 22:263; scRepertoire quietTCRgenes), TRAC/TRBC kept
    "^TRAV","^TRBV","^TRGV","^TRDV","^TRAJ","^TRBJ","^TRDJ","^TRBD","^TRDD",
    #BCR V/J segments and immunoglobulin
    "^IGKV.*","^IGKJ.*","^IGHV.*","^IGHJ.*","^IGLJ.*","^IGLV.*",
    #ribosomal (Baran 2019)
    "^RPS.*","^RPL.*","^RPP.*",
    #heat-shock family 
    "^HSPA.*","^HSPB.*",
]

Noisy_gnames = STRESS + ["JCHAIN","HLA-A","HLA-B","HLA-C","HLA-E","HLA-DRB1","HLA-DQA1","HLA-DQB1"]
Noisy_gpatterns = ["^IGHM.*","^IGHA.*","^IGHG.*","^IGKV.*","^IGKJ.*","^IGHV.*","^IGHJ.*","^IGLJ.*","^IGLV.*","^HSPA.*","^HSPB.*"]

mc.pl.exclude_genes(
    adata=merged,
    excluded_gene_names=Excluded_gnames,
    excluded_gene_patterns=Excluded_gpatterns,
    random_seed=RANDOM_SEED,
)
print("Excluded genes:", int(merged.var["excluded_gene"].sum()))


## 6. Combine all cell filters, then extract clean data

`exclude_cells` is run with a permissive UMI range (so it mainly applies the
excluded-gene-fraction filter), and the MAD + miQC masks are OR-ed into `excluded_cell`. This keeps
the composite excluded-fraction safety net while making the count/mito thresholds data-driven.


In [ ]:
# explicit masks
excl_gene_mask = merged.var["excluded_gene"].values
excl_umis = np.asarray(merged[:, excl_gene_mask].X.sum(axis=1)).flatten()
merged.obs["excluded_gene_fraction"] = excl_umis / np.maximum(merged.obs["total_umis"].values, 1)
low_umi_any   = merged.obs["total_umis"].values < MIN_UMI_FLOOR
high_frac_any = merged.obs["excluded_gene_fraction"].values > MAX_EXCL_FRAC

# metacell filter
mc.pl.exclude_cells(
    adata=merged,
    properly_sampled_min_cell_total=MIN_UMI_FLOOR, #low UMI floor as safety
    properly_sampled_max_cell_total=10_000_000, #Very high cutoff, MAD would kick in before this anyway
    properly_sampled_max_excluded_genes_fraction=MAX_EXCL_FRAC, #
)
mc_excluded = merged.obs["excluded_cell"].values.copy()
#mad  = merged.obs["mad_outlier"].values
#miqc = merged.obs["miqc_compromised"].values

# final union actually used to filter
merged.obs["excluded_cell"] = mc_excluded  #| miqc | mad

# per-sample, per-stage attrition (sequential: mad -> miqc -> metacells)
aud = pd.DataFrame({
    "sample_origin": merged.obs["sample_origin"].values,
    #"lost_mad":            mad,
    #"lost_miqc_new":       miqc & ~mad,
    "lost_metacells":  mc_excluded, #& ~mad & ~miqc,
    "removed":             merged.obs["excluded_cell"].values,
    #"mad_flagged":         mad,
    #"miqc_flagged":        miqc,
    "low_umi_flagged":     low_umi_any,
    "excl_frac_flagged":   high_frac_any,
})
g = aud.groupby("sample_origin", observed=True)
attrition = pd.DataFrame({
    "n_start":            g.size(),
    #"lost_mad":           g["lost_mad"].sum(),
    #"lost_miqc_new":      g["lost_miqc_new"].sum(),
    "lost_metacells":     g["lost_metacells"].sum(),
    "n_removed_total":    g["removed"].sum(),
    #"mad_flagged":        g["mad_flagged"].sum(),
    #"miqc_flagged":       g["miqc_flagged"].sum(),
    "low_umi_flagged":    g["low_umi_flagged"].sum(),
    "excl_frac_flagged":  g["excl_frac_flagged"].sum(),
})
attrition["n_clean"] = attrition["n_start"] - attrition["n_removed_total"]
attrition = attrition.reset_index()
print(attrition.to_string(index=False))

clean = mc.pl.extract_clean_data(adata=merged)
print(f"\nCells: {merged.n_obs} -> {clean.n_obs} ({clean.n_obs/merged.n_obs:.2%})")
print(f"Genes: {merged.n_vars} -> {clean.n_vars} ({clean.n_vars/merged.n_vars:.2%})")


## 7. Mark lateral & noisy genes on the clean object


In [ ]:
mc.pl.mark_lateral_genes(
    adata=clean, lateral_gene_names=Lateral_gnames, lateral_gene_patterns=Lateral_gpatterns,
)
mc.pl.mark_noisy_genes(
    adata=clean, noisy_gene_names=Noisy_gnames, noisy_gene_patterns=Noisy_gpatterns,
)
print("lateral:", int(clean.var["lateral_gene"].sum()), "| noisy:", int(clean.var["noisy_gene"].sum()))


## 8. Select feature genes + QC that no lateral gene leaked in

The metacells docs recommend checking the `selected_gene` mask against `lateral_gene`: any lateral
gene that got selected would degrade the metacells and should be added to the lateral list and the
run repeated.


In [ ]:
selected = mc.pl.extract_selected_data(
    adata=clean, min_gene_relative_variance=None, random_seed=RANDOM_SEED,
)
print("selected feature genes:", int(clean.var["selected_gene"].sum()))

leaked = clean.var_names[(clean.var["selected_gene"].values) &
                         (clean.var.get("lateral_gene", pd.Series(False, index=clean.var_names)).values)]
print("Lateral genes that leaked into selection (should be empty):", list(leaked))


## 9. Target metacell size & pile size, then run DAC

`TARGET_METACELL_SIZE` / `TARGET_METACELL_UMIS` being passed at top of notebook


In [ ]:
mc.pl.compute_target_pile_size(
    adata=clean,
    target_metacell_size=TARGET_METACELL_SIZE,
    target_metacell_umis=TARGET_METACELL_UMIS,
    min_target_pile_size=MIN_PILE,
    max_target_pile_size=MAX_PILE,
    target_metacells_in_pile=TARGET_METACELLS_IN_PILE,
)

with mc.ut.progress_bar():
    mc.pl.divide_and_conquer_pipeline(
        adata=clean,
        target_metacell_size=TARGET_METACELL_SIZE,
        target_metacell_umis=TARGET_METACELL_UMIS,
        min_target_pile_size=MIN_PILE,
        max_target_pile_size=MAX_PILE,
        target_metacells_in_pile=TARGET_METACELLS_IN_PILE,
        random_seed=RANDOM_SEED,
    )


## 10. Collect metacells + outlier QC


In [ ]:
metacells = mc.pl.collect_metacells(clean, name=f"{PROJECT}.metacells", random_seed=RANDOM_SEED)

# per-sample outlier stage (metacell < 0), appended onto the attrition table
grp = clean.obs.groupby("sample_origin", observed=True)["metacell"]
outdf = pd.DataFrame({
    "n_clean_cells": grp.size(),
    "n_outlier": grp.apply(lambda s: int((np.asarray(s) < 0).sum())),
}).reset_index()
outdf["n_in_metacell"] = outdf["n_clean_cells"] - outdf["n_outlier"]
outdf["outlier_pct"] = (100 * outdf["n_outlier"] / outdf["n_clean_cells"]).round(2)

excluded_per_sample = attrition.merge(outdf, on="sample_origin", how="left")
write_xlsx_sheets(QC_XLSX_PATH, {"excluded_per_sample": excluded_per_sample})
print(excluded_per_sample.to_string(index=False))

n_outlier = int((clean.obs["metacell"] < 0).sum())
print(f"\nMetacells: {metacells.n_obs} | outlier cells: {n_outlier} "
      f"({100*n_outlier/clean.n_obs:.2f}%)")


## 11. Rare gene modules 

pulling these lists helps identify rare cell types and also flags potential genes to be moved to lateral lists


In [ ]:
#what the rare gene detector actually pulls
rare_var_cols = [c for c in clean.var.columns if "rare" in c.lower()]
rare_obs_cols = [c for c in clean.obs.columns if "rare" in c.lower()]
print("rare-related var columns:", rare_var_cols)
print("rare-related obs columns:", rare_obs_cols)

if "rare_gene" in clean.var:
    print("genes in ANY rare module  (var['rare_gene']):", int(clean.var["rare_gene"].sum()))
if "rare_gene_module" in clean.var:
    v = pd.Series(clean.var["rare_gene_module"]).astype(int)
    print("distinct gene-module indices (var, -1 = none):", sorted(v.unique()))
if "rare_gene_module" in clean.obs:
    c = pd.Series(clean.obs["rare_gene_module"]).astype(int)
    print("cells per rare module (obs, -1 = none):")
    print(c.value_counts().sort_index())
print("rare-related metacells.obs columns:", [x for x in metacells.obs.columns if "rare" in x.lower()])

# per-module cell counts: the per-cell rare index is stored under a different name than the per-gene one
def _find_col(df, names):
    for nm in names:
        if nm in df.columns:
            return nm
    return None

_cell_names = ["rare_gene_module", "cells_rare_gene_module", "rare_cell_module", "cell_rare_gene_module"]
cells_per_module = {}
obs_col = _find_col(clean.obs, _cell_names)
mc_col  = _find_col(metacells.obs, _cell_names)
if obs_col is not None:
    s = pd.Series(clean.obs[obs_col].values).astype(int)
    cells_per_module = {int(m): int((s == m).sum()) for m in s.unique() if m >= 0}
    _src = f"clean.obs['{obs_col}']"
elif mc_col is not None and "grouped" in metacells.obs.columns:
    mm  = pd.Series(metacells.obs[mc_col].values).astype(int)
    grp = pd.Series(metacells.obs["grouped"].values)
    cells_per_module = {int(m): int(grp[mm == m].sum()) for m in mm.unique() if m >= 0}
    _src = f"metacells.obs['{mc_col}'] x grouped"
else:
    _src = "unavailable (n_cells will be NaN)"
print("per-module cell-count source:", _src)

# ---- 2) build the summary from whichever schema is present ----
rows_summary, rows_long = [], []
gmod = pd.Series(clean.var["rare_gene_module"].values, index=clean.var_names).astype(int) \
       if "rare_gene_module" in clean.var else None
per_module_cols = sorted([c for c in clean.var.columns if c.startswith("rare_gene_module_")],
                         key=lambda s: int(s.rsplit("_", 1)[1]))

if gmod is not None and (gmod >= 0).any():                 # integer-index schema (expected)
    for m in sorted(x for x in gmod.unique() if x >= 0):
        genes = gmod.index[gmod == m].tolist()
        rows_summary.append({"module": m, "n_genes": len(genes),
                             "n_cells": cells_per_module.get(int(m), np.nan), "genes": ", ".join(genes)})
        rows_long += [{"module": m, "gene": g} for g in genes]
elif per_module_cols:                                      # per-module boolean-mask schema
    for col in per_module_cols:
        m = int(col.rsplit("_", 1)[1]); genes = clean.var_names[clean.var[col].values].tolist()
        rows_summary.append({"module": m, "n_genes": len(genes),
                             "n_cells": cells_per_module.get(m, np.nan), "genes": ", ".join(genes)})
        rows_long += [{"module": m, "gene": g} for g in genes]

summary_df = pd.DataFrame(rows_summary, columns=["module", "n_genes", "n_cells", "genes"])
long_df    = pd.DataFrame(rows_long, columns=["module", "gene"])
if len(summary_df):
    print(f"\n{len(summary_df)} rare gene modules")
    display(summary_df)
else:
    print("\n>>> No rare gene modules were detected in this run.\n")
write_xlsx_sheets(QC_XLSX_PATH, {"rare_gene_modules": long_df, "rare_gene_summary": summary_df})


## 12. Save both objects

`clean_cells.h5ad` carries the per-cell metacell assignment plus the `lateral_gene` / `noisy_gene`
/ `rare_gene` masks ,  the 03 script reads those to (a) exclude lateral genes from module HVGs and
(b) pull rare genes. `metacells.h5ad` is the collected metacell object.


In [ ]:
# clean_cells carries the per-cell metacell assignment and the gene masks,
# metacells is the collected metacell object. 03 reads both.
clean.write_h5ad(os.path.join(h5ad_dir, f"{PROJECT}.clean_cells.h5ad"))
metacells.write_h5ad(os.path.join(h5ad_dir, f"{PROJECT}.metacells.h5ad"))
print("Saved to", h5ad_dir)
